# 18 — Extract stored figures from every notebook

Harvests every image embedded in the executed notebooks into `output/figures/from_notebooks/`,
named `<notebook>_cellNN_imgK.png` — flat in one folder, no per-notebook subfolders (most notebooks
embed no images, so subfolders were mostly empty; flattened 2026-07-30 on user instruction) — with a
manifest mapping each file back to its notebook, cell index and the cell's stated `# Goal:` line.
Nothing is executed or recomputed — the pngs are the stored outputs of each notebook's last headless
run. `nbconvert --inplace` keeps only that latest run, so this is the current stored state, not an
archive of past runs; re-running this notebook after re-running any other refreshes its images.

Numbering note: this tooling notebook took 18 while 15 and 17 were still reserved; both now exist
(15 = frozen agent rules, 17 = the sealed-year one-shot).

In [1]:
# Goal: one png on disk per image embedded in any executed notebook, plus a manifest row for each.
import json                                  # notebooks are json files; cells and outputs are plain dicts
import base64                                # embedded pngs are base64 text - decoded back to bytes here
import pandas as pd                          # the manifest table written at the end
from pathlib import Path

SRC = Path("../src")
OUT = Path("../output/figures/from_notebooks"); OUT.mkdir(parents=True, exist_ok=True)

rows = []                                    # manifest: one row per extracted image
for nb_path in sorted(SRC.glob("*.ipynb")):
    if nb_path.stem == "18_extract_figures":  # skip self - its on-disk state predates this run
        continue
    cells = json.loads(nb_path.read_text(encoding="utf-8"))["cells"]
    n_before = len(rows)
    for ci, cell in enumerate(cells):
        if cell["cell_type"] != "code":       # only code cells carry outputs
            continue
        # the cell's declared purpose, kept in the manifest so a png is traceable without opening the notebook
        first = cell["source"][0].strip() if cell["source"] else ""
        goal = first[len("# Goal:"):].strip() if first.startswith("# Goal:") else ""
        for oi, out in enumerate(cell.get("outputs", [])):
            png = out.get("data", {}).get("image/png")   # stream/error outputs have no data dict -> None
            if png is None:
                continue
            f = OUT / f"{nb_path.stem}_cell{ci:02d}_img{oi}.png"   # notebook stem + cell index keep the
            f.write_bytes(base64.b64decode(png))                   # flat names unique and ordered
            rows.append({"notebook": nb_path.stem, "cell_index": ci, "goal": goal, "file": f.name})
    print(f"{nb_path.stem}: {len(rows) - n_before} images")

01_fetch_dataset: 0 images
02_eda_dataset: 5 images
03_provider_flags: 0 images
04_baseline_forecasting: 0 images
05_kh03_beds: 1 images
06_merge_canonical: 0 images
07_beat_persistence: 0 images
08_part3_flows: 0 images
09_longer_horizons: 0 images
10_agent_logic: 0 images
11_federated_learning: 0 images
12_coarser_grain: 0 images
13_report_figures: 13 images
14_fl_scaleup: 0 images
15_agent_rules_frozen: 0 images
16_fl_neural_clients: 1 images
17_sealed_year_eval: 0 images
19_neural_c110_multiseed: 0 images


In [2]:
# Goal: from_notebooks/manifest.csv - the file -> (notebook, cell, goal) map for every harvested png.
man = pd.DataFrame(rows)
assert len(man) > 0                          # zero images means the src notebooks lost their stored outputs
man.to_csv(OUT / "manifest.csv", index=False)
print(len(man), "images from", man["notebook"].nunique(), "notebooks ->", OUT)
print(man.groupby("notebook").size().to_string())

20 images from 4 notebooks -> ..\output\figures\from_notebooks
notebook
02_eda_dataset           5
05_kh03_beds             1
13_report_figures       13
16_fl_neural_clients     1
